# Poziom 5 - Podstawy Sparka

In [1]:
import os

os.makedirs("output", exist_ok=True)

## Część 1

Zainicjuj SparkSession:

In [2]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .appName("spark_test") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/06/05 22:17:12 WARN Utils: Your hostname, Mikoajs-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.50 instead (on interface en0)
25/06/05 22:17:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/05 22:17:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Sprawdź wersję Spark:

In [3]:
print("Spark version:", spark.version)

Spark version: 4.0.0


Stwórz prosty RDD i zbierz dane:

In [4]:
data = [1, 2, 3, 4, 5]

rdd = spark.sparkContext.parallelize(data)

print("RDD collect:", rdd.collect())

RDD collect: [1, 2, 3, 4, 5]


Stwórz DataFrame i wyświetl dane:

In [5]:
df = spark.createDataFrame([("Andorra", 98), ("United Arab Emirates", 1683)], ["country_name", "total_confirmed"])

df.show()

+--------------------+---------------+
|        country_name|total_confirmed|
+--------------------+---------------+
|             Andorra|             98|
|United Arab Emirates|           1683|
+--------------------+---------------+



Opcjonalnie zatrzymaj sesję Spark:

In [6]:
spark.stop()

## Część 2

#### **2.1.** Wczytaj dane ze swojego pliku CSV dotyczącego danych COVID-19 (pamiętaj o nagłówku dla kolumn).

In [7]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.driver.extraJavaOptions", "-Djava.security.manager=allow") \
    .config("spark.executor.extraJavaOptions", "-Djava.security.manager=allow") \
    .appName("covid_data_analysis") \
    .getOrCreate()


In [9]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

schema = StructType([
    StructField("", IntegerType(), True),
    StructField("location_key", StringType(), True),
    StructField("date", StringType(), True),
    StructField("country_name", StringType(), True),
    StructField("aggregation_level", IntegerType(), True),
    StructField("iso_3166_1_alpha_2", StringType(), True),
    StructField("iso_3166_1_alpha_3", StringType(), True),
    StructField("subregion1_code", StringType(), True),
    StructField("subregion1_name", StringType(), True),
    StructField("subregion2_code", StringType(), True),
    StructField("subregion2_name", StringType(), True),
    StructField("population", IntegerType(), True),
    StructField("gdp_usd", FloatType(), True),
    StructField("nurses_per_1000", FloatType(), True),
    StructField("physicians_per_1000", FloatType(), True),
    StructField("hospital_beds_per_1000", FloatType(), True),
    StructField("health_expenditure_usd", FloatType(), True),
    StructField("new_confirmed", IntegerType(), True),
    StructField("cumulative_confirmed", IntegerType(), True),
    StructField("new_confirmed_male", IntegerType(), True),
    StructField("new_confirmed_female", IntegerType(), True),
    StructField("cumulative_confirmed_male", IntegerType(), True),
    StructField("cumulative_confirmed_female", IntegerType(), True),
    StructField("new_deceased", IntegerType(), True),
    StructField("cumulative_deceased", IntegerType(), True),
    StructField("new_deceased_male", IntegerType(), True),
    StructField("new_deceased_female", IntegerType(), True),
    StructField("cumulative_deceased_male", IntegerType(), True),
    StructField("cumulative_deceased_female", IntegerType(), True),
    StructField("new_persons_vaccinated", IntegerType(), True),
    StructField("cumulative_persons_vaccinated", IntegerType(), True),
    StructField("new_persons_fully_vaccinated", IntegerType(), True),
    StructField("cumulative_persons_fully_vaccinated", IntegerType(), True),
    StructField("new_vaccine_doses_administered", IntegerType(), True),
    StructField("cumulative_vaccine_doses_administered", IntegerType(), True),
    StructField("search_trends_alcoholism", FloatType(), True),
    StructField("search_trends_anxiety", FloatType(), True),
    StructField("search_trends_depression", FloatType(), True),
    StructField("search_trends_insomnia", FloatType(), True),
    StructField("country_population_rank", FloatType(), True),
    StructField("capital", StringType(), True),
    StructField("continent", StringType(), True),
    StructField("country_population_2022", FloatType(), True),
    StructField("country_population_2020", FloatType(), True),
    StructField("country_population_2015", FloatType(), True),
    StructField("country_population_2010", FloatType(), True),
    StructField("country_area_sq_km", FloatType(), True),
    StructField("country_population_density_per_sq_km", FloatType(), True),
    StructField("country_population_growth_rate", FloatType(), True),
    StructField("country_world_population_percentage", FloatType(), True),
    StructField("country_gdp_usd_2010", FloatType(), True),
    StructField("country_gdp_usd_2011", FloatType(), True),
    StructField("country_gdp_usd_2012", FloatType(), True),
    StructField("country_gdp_usd_2013", FloatType(), True),
    StructField("country_gdp_usd_2014", FloatType(), True),
    StructField("country_gdp_usd_2015", FloatType(), True),
    StructField("country_gdp_usd_2016", FloatType(), True),
    StructField("country_gdp_usd_2017", FloatType(), True),
    StructField("country_gdp_usd_2018", FloatType(), True),
    StructField("country_gdp_usd_2019", FloatType(), True),
    StructField("country_gdp_usd_2020", FloatType(), True),
    StructField("country_gdp_usd_2021", FloatType(), True),
    StructField("country_gdp_usd_2022", FloatType(), True),
    StructField("country_gdp_usd_2023", FloatType(), True),
    StructField("aqi_daily", FloatType(), True),
    StructField("aqi_defining_parameter", StringType(), True),
    StructField("country_electricity_demand_twh_monthly", FloatType(), True),
    StructField("country_electricity_generation_twh_monthly", FloatType(), True)
])


df_combined = spark.read.csv("output/combined_part_7.csv", header=True, schema=schema)
df_combined = df_combined.filter(df_combined.aggregation_level == 0)

df_combined.show(5)

25/06/05 22:17:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+------------+----------+------------+-----------------+------------------+------------------+---------------+---------------+---------------+---------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+------------+-------------------+-----------------+-------------------+------------------------+--------------------------+----------------------+-----------------------------+----------------------------+-----------------------------------+------------------------------+-------------------------------------+------------------------+---------------------+------------------------+----------------------+-----------------------+----------------+---------+-----------------------+-----------------------+-----------------------+-----------------------+------------------+----------------------------------

#### **2.2.** Utwórz DataFrame. Pamiętaj o utworzeniu własnego schematu z wybranymi kolumnami.

In [10]:
selected_columns = [
    "date", "country_name", "population",
    "gdp_usd", "nurses_per_1000", "physicians_per_1000",
    "hospital_beds_per_1000", "health_expenditure_usd", "new_confirmed",
    "cumulative_confirmed", "new_deceased", "cumulative_deceased",
    "new_persons_vaccinated", "cumulative_persons_vaccinated", "search_trends_depression",
    "continent", "country_area_sq_km", "country_population_density_per_sq_km",
    "country_gdp_usd_2010", "country_gdp_usd_2011", "country_gdp_usd_2012",
    "country_gdp_usd_2013", "country_gdp_usd_2014", "country_gdp_usd_2015",
    "country_gdp_usd_2016", "country_gdp_usd_2017", "country_gdp_usd_2018",
    "country_gdp_usd_2019", "country_gdp_usd_2020", "country_gdp_usd_2021",
    "country_gdp_usd_2022", "country_gdp_usd_2023",
]

df_combined = df_combined.select(*selected_columns)

df_combined.printSchema()

root
 |-- date: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- population: integer (nullable = true)
 |-- gdp_usd: float (nullable = true)
 |-- nurses_per_1000: float (nullable = true)
 |-- physicians_per_1000: float (nullable = true)
 |-- hospital_beds_per_1000: float (nullable = true)
 |-- health_expenditure_usd: float (nullable = true)
 |-- new_confirmed: integer (nullable = true)
 |-- cumulative_confirmed: integer (nullable = true)
 |-- new_deceased: integer (nullable = true)
 |-- cumulative_deceased: integer (nullable = true)
 |-- new_persons_vaccinated: integer (nullable = true)
 |-- cumulative_persons_vaccinated: integer (nullable = true)
 |-- search_trends_depression: float (nullable = true)
 |-- continent: string (nullable = true)
 |-- country_area_sq_km: float (nullable = true)
 |-- country_population_density_per_sq_km: float (nullable = true)
 |-- country_gdp_usd_2010: float (nullable = true)
 |-- country_gdp_usd_2011: float (nullable = true)
 |-- 

#### **2.3.** Wyświetl co najmniej 5 kolumn i wyjaśnij ich znaczenie dla analizy przypadków COVID-19.

In [11]:
df_combined.select('date', 'country_name', 'new_confirmed', 'new_deceased', 'new_persons_vaccinated').show(5)

+----------+------------+-------------+------------+----------------------+
|      date|country_name|new_confirmed|new_deceased|new_persons_vaccinated|
+----------+------------+-------------+------------+----------------------+
|2020-01-01|     Andorra|            0|           0|                  NULL|
|2020-01-02|     Andorra|            0|           0|                  NULL|
|2020-01-03|     Andorra|            0|           0|                  NULL|
|2020-01-04|     Andorra|            0|           0|                  NULL|
|2020-01-05|     Andorra|            0|           0|                  NULL|
+----------+------------+-------------+------------+----------------------+
only showing top 5 rows


#### **2.4.** Dodaj kolumnę, która będzie zbierała dane, które uznasz za stosowne i które są istotne z punktu widzenia analizy COVID-19 (np. wyliczenie gęstości zaludnienia na podstawie liczby ludności i powierzchni kraju).

In [12]:
df_combined = df_combined.withColumn(
    "percent_population_vaccinated",
    (df_combined["cumulative_persons_vaccinated"] / df_combined["population"]) * 100
)

df_combined.show(5)

+----------+------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+
|      date|country_name|population|   gdp_usd|nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_deceased|cumulative_deceased|new_persons_vaccinated|cumulative_persons_vaccinated|search_trends_depression|continent|country_area_sq_km|country_population_density_per

#### **2.5.** Połącz wybrane kolumny ze sobą w sposób niosący nowe informacje za pomocą funkcji concat.

In [13]:
df_combined = df_combined.withColumn(
    "country_gdp_usd_2010_to_2023",
    F.concat_ws(
        ",",
        df_combined["country_gdp_usd_2010"],
        df_combined["country_gdp_usd_2011"],
        df_combined["country_gdp_usd_2012"],
        df_combined["country_gdp_usd_2013"],
        df_combined["country_gdp_usd_2014"],
        df_combined["country_gdp_usd_2015"],
        df_combined["country_gdp_usd_2016"],
        df_combined["country_gdp_usd_2017"],
        df_combined["country_gdp_usd_2018"],
        df_combined["country_gdp_usd_2019"],
        df_combined["country_gdp_usd_2020"],
        df_combined["country_gdp_usd_2021"],
        df_combined["country_gdp_usd_2022"],
        df_combined["country_gdp_usd_2023"]
    )
)

df_combined.show(5, truncate=False)

+----------+------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd   |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_de

#### **2.6.** Wyfiltruj co najmniej 10 wybranych przez siebie, istotnych informacji (np. kraj z liczbą łóżek większą od 10000 czy dzień, w którym było najmniej zachorowań w danym kraju).

##### **1.** Kraje z powierzchnią większą niż 1 000 000 km²:

In [14]:
df_large_countries = df_combined.filter(
    df_combined["country_area_sq_km"] > 1_000_000
)

df_large_countries.show(5, truncate=False)

df_large_countries.write.mode("overwrite").parquet(
    "output/large_countries.parquet"
)

+----------+------------+----------+-----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd    |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_con

##### **2.** Kraje z gęstością zaludnienia większą niż 100 osób na km²:

In [15]:
df_high_density_countries = df_combined.filter(
    df_combined["country_population_density_per_sq_km"] > 100
)

df_high_density_countries.show(5, truncate=False)

df_high_density_countries.write.mode("overwrite").parquet(
    "output/high_density_countries.parquet"
)

+----------+------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd   |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_de

##### **3.** Sezon grypowy 2020/21 w USA (październik 2020 - kwiecień 2021):

In [16]:
df_flu_season_usa = df_combined.filter(
    (df_combined["country_name"] == "United States of America") &
    (df_combined["date"].between("2020-10-01", "2021-04-30"))
)

df_flu_season_usa.show(5, truncate=False)

df_flu_season_usa.write.mode("overwrite").parquet(
    "output/flu_season_usa.parquet"
)

+----------+------------------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+-------------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name            |population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expendit

##### **4.** Kraje europejskie:

In [17]:
df_european_countries = df_combined.filter(
    df_combined["continent"] == "Europe"
)

df_european_countries.show(5, truncate=False)

df_european_countries.write.mode("overwrite").parquet(
    "output/european_countries.parquet"
)

+----------+------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd   |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_de

##### **5.** Kraje z wysokim wydatkiem na ochronę zdrowia (powyżej 5000 USD na osobę):

In [18]:
df_high_health_expenditure_countries = df_combined.filter(
    df_combined["health_expenditure_usd"] > 5000
)

df_high_health_expenditure_countries.show(5, truncate=False)

df_high_health_expenditure_countries.write.mode("overwrite").parquet(
    "output/high_health_expenditure_countries.parquet"
)

+----------+------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulati

##### **6.** Kraje bogato wyposażone w łóżka szpitalne (więcej niż 5 łóżek na 1000 mieszkańców):

In [19]:
df_hospital_beds_countries = df_combined.filter(
    df_combined["hospital_beds_per_1000"] > 5
)

df_hospital_beds_countries.show(5, truncate=False)

df_hospital_beds_countries.write.mode("overwrite").parquet(
    "output/hospital_beds_countries.parquet"
)

+----------+--------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name  |population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|c

##### **7.** Kraje z bogatym personalem medycznym (więcej niż 5 lekarzy i 5 pielęgniarek na 1000 mieszkańców):

In [20]:
df_medical_staff_countries = df_combined.filter(
    (df_combined["physicians_per_1000"] > 5) &
    (df_combined["nurses_per_1000"] > 5)
)

df_medical_staff_countries.show(5, truncate=False)

df_medical_staff_countries.write.mode("overwrite").parquet(
    "output/medical_staff_countries.parquet"
)

+----------+------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulati

##### **8.** Rekordy z niezerowymi wartościami nowych szczepień:

In [21]:
df_nonzero_vaccination = df_combined.filter(
    df_combined["new_persons_vaccinated"] > 0
)

df_nonzero_vaccination.show(5, truncate=False)

df_nonzero_vaccination.write.mode("overwrite").parquet(
    "output/nonzero_vaccination.parquet"
)

+----------+------------+----------+----------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd   |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_de

##### **9.** Rekordy z wysokimi wartościami nowych zachorowań (powyżej 1000):

In [22]:
df_high_new_cases = df_combined.filter(
    df_combined["new_confirmed"] > 1000
)

df_high_new_cases.show(5, truncate=False)

df_high_new_cases.write.mode("overwrite").parquet(
    "output/high_new_cases.parquet"
)

+----------+--------------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+---------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name        |population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confi

##### **10.** Rekordy z wysokimi wartościami nowych zgonów (powyżej 100):

In [23]:
df_high_new_deceased = df_combined.filter(
    df_combined["new_deceased"] > 100
)

df_high_new_deceased.show(5, truncate=False)

df_high_new_deceased.write.mode("overwrite").parquet(
    "output/high_new_deceased.parquet"
)

+----------+------------+----------+------------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------+-------------------+----------------------+-----------------------------+------------------------+-------------+------------------+------------------------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|date      |country_name|population|gdp_usd     |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumu

#### **2.7.** Wykonaj 5 agregacji na swoim DF (np. min, max, agg, count, mean) uprzednio pogrupowawszy kolumny (groupBy).

##### **1.** Data pojawienia się pierwszych zachorowań w każdym kraju (min date):

In [24]:
df_first_cases = df_combined.filter(
    df_combined["new_confirmed"] > 0
).groupBy("country_name").agg(
    F.min("date").alias("first_cases_date")
)

df_first_cases.show(10, truncate=False)

df_first_cases.write.mode("overwrite").parquet(
    "output/first_cases_date.parquet"
)

+-------------------+----------------+
|country_name       |first_cases_date|
+-------------------+----------------+
|Afghanistan        |2020-02-22      |
|Albania            |2020-03-07      |
|Algeria            |2020-01-02      |
|American Samoa     |2021-09-17      |
|Andorra            |2020-03-01      |
|Angola             |2020-01-02      |
|Anguilla           |2020-03-25      |
|Antigua and Barbuda|2020-03-13      |
|Argentina          |2020-01-01      |
|Armenia            |2020-02-29      |
+-------------------+----------------+
only showing top 10 rows


##### **2.** Maksymalna liczba śmierci w każdym kraju (max new_deceased):

In [25]:
df_max_deceased = df_combined.filter(
    df_combined["new_deceased"] > 0
).groupby("country_name").agg(
    F.max("new_deceased").alias("max_new_deceased")
)

df_max_deceased.show(10, truncate=False)

df_max_deceased.write.mode("overwrite").parquet(
    "output/max_new_deceased.parquet"
)

+--------------------+----------------+
|country_name        |max_new_deceased|
+--------------------+----------------+
|Anguilla            |1               |
|Afghanistan         |159             |
|Argentina           |656             |
|Angola              |26              |
|Albania             |21              |
|Andorra             |6               |
|Armenia             |70              |
|United Arab Emirates|20              |
|Antigua and Barbuda |12              |
|Belgium             |324             |
+--------------------+----------------+
only showing top 10 rows


##### **3.** Liczba dni z niezerową liczbą nowych szczepień w każdym kraju (count new_persons_vaccinated):

In [26]:
df_nonzero_vaccination_days = df_combined.filter(
    df_combined["new_persons_vaccinated"] > 0
).groupBy("country_name").agg(
    F.count("date").alias("nonzero_vaccination_days")
)

df_nonzero_vaccination_days.show(10, truncate=False)

df_nonzero_vaccination_days.write.mode("overwrite").parquet(
    "output/nonzero_vaccination_days.parquet"
)

+--------------------+------------------------+
|country_name        |nonzero_vaccination_days|
+--------------------+------------------------+
|Anguilla            |47                      |
|Afghanistan         |67                      |
|Argentina           |623                     |
|Angola              |72                      |
|Albania             |212                     |
|Andorra             |46                      |
|Armenia             |44                      |
|United Arab Emirates|125                     |
|Antigua and Barbuda |89                      |
|Belgium             |608                     |
+--------------------+------------------------+
only showing top 10 rows


##### **4.** Suma śmierci na każdym kontynencie (sum new_deceased):

In [27]:
df_sum_deceased_by_continent = df_combined.filter(
    (df_combined["new_deceased"] > 0) &
    (df_combined["continent"].isNotNull())
).groupBy("continent").agg(
    F.sum("new_deceased").alias("total_new_deceased")
)

df_sum_deceased_by_continent.show(10, truncate=False)

df_sum_deceased_by_continent.write.mode("overwrite").parquet(
    "output/sum_deceased_by_continent.parquet"
)

+-------------+------------------+
|continent    |total_new_deceased|
+-------------+------------------+
|Europe       |1892120           |
|Africa       |258237            |
|North America|1448467           |
|South America|1351494           |
|Asia         |1494519           |
|Oceania      |20646             |
+-------------+------------------+



##### **5.** Średnie trendy wyszukiwania frazy "depression" w każdym kraju (mean search_trend_depression):

In [28]:
df_mean_search_trend_depression = df_combined.filter(
    df_combined["search_trends_depression"].isNotNull()
).groupBy("country_name").agg(
    F.mean("search_trends_depression").alias("mean_search_trends_depression")
)

df_mean_search_trend_depression.show(10, truncate=False)

df_mean_search_trend_depression.write.mode("overwrite").parquet(
    "output/mean_search_trends_depression.parquet"
)

+------------------------+-----------------------------+
|country_name            |mean_search_trends_depression|
+------------------------+-----------------------------+
|Australia               |6.419503042228807            |
|United Kingdom          |5.197636907531089            |
|Ireland                 |13.375791063173306           |
|New Zealand             |7.113356996017828            |
|Singapore               |4.880953356412071            |
|United States of America|4.36600406271449             |
+------------------------+-----------------------------+



#### **2.8.** Dodaj kolumnę z datą przekształconą do typu DateType. Użyj funkcji to_date(), aby utworzyć dwie nowe kolumny: month i year, korzystając z funkcji month() i year(). Wykonaj agregację pokazującą, jak zmieniała się liczba przypadków COVID-19 w czasie (np. suma przypadków w danym miesiącu lub roku).

In [29]:
df_combined = df_combined.withColumn("datetype_date", F.to_date(df_combined["date"]))
df_combined = df_combined.withColumn("month", F.month(df_combined["datetype_date"]))
df_combined = df_combined.withColumn("year", F.year(df_combined["datetype_date"]))

In [30]:
df_monthly_cases = df_combined.groupBy("year", "month").agg(
    F.sum("new_confirmed").alias("total_new_confirmed"),
)

df_monthly_cases.sort("year", "month").show(10)

df_monthly_cases.write.mode("overwrite").parquet(
    "output/monthly_cases.parquet"
)

+----+-----+-------------------+
|year|month|total_new_confirmed|
+----+-----+-------------------+
|2020|    1|              24234|
|2020|    2|              76098|
|2020|    3|             810784|
|2020|    4|            2236202|
|2020|    5|            2890433|
|2020|    6|            4351393|
|2020|    7|            7314027|
|2020|    8|            8156365|
|2020|    9|            8798198|
|2020|   10|           12441800|
+----+-----+-------------------+
only showing top 10 rows


#### **2.9.** Połącz za pomocą wszystkich typów złączeń dwa DF wybrane przez Ciebie w poprzednich zadaniach w jeden nowy DF. Dla każdego typu JOINa, możesz wybrać inną parę DF. Wyświetl nowe DFy.

Każde działanie udowodnij używając .show(). Pamiętaj o zapisaniu swoich DF do wybranego formatu pliku (CSV, JSON, PARQUET, AVRO).

In [31]:
df_countries = spark.read.csv("output/countries.csv", header=True, inferSchema=True).drop("_c0")
df_cases = spark.read.csv("output/cases.csv", header=True, inferSchema=True).drop("_c0")
df_deceased = spark.read.csv("output/deceased.csv", header=True, inferSchema=True).drop("_c0")
df_vaccinations = spark.read.csv("output/vaccinations.csv", header=True, inferSchema=True).drop("_c0")
df_disorders = spark.read.csv("output/disorders.csv", header=True, inferSchema=True).drop("_c0")

In [32]:
def drop_null_rows_except_date_location(df):
    cols_to_check = [col for col in df.columns if col not in ['date', 'location_key']]
    df_filtered = df.filter(
        ~(sum(F.col(col).isNull().cast("int") for col in cols_to_check) == len(cols_to_check))
    )
    return df_filtered


def add_col_prefix(df, prefix):
    return df.select(
        *[F.col(col).alias(f"{prefix}_{col}") for col in df.columns]
    )

In [33]:
df_countries = drop_null_rows_except_date_location(df_countries)
df_cases = drop_null_rows_except_date_location(df_cases)
df_deceased = drop_null_rows_except_date_location(df_deceased)
df_vaccinations = drop_null_rows_except_date_location(df_vaccinations)
df_disorders = drop_null_rows_except_date_location(df_disorders)

##### **1.** Inner Join:

In [34]:
df_inner_join_cases_countries = df_cases.join(
    df_countries, on=["date", "location_key"], how="inner"
)

df_inner_join_cases_countries.show(10, truncate=False)

df_inner_join_cases_countries.write.mode("overwrite").parquet(
    "output/inner_join_cases_countries.parquet"
)

+----------+------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+------------+-----------------+------------------+------------------+---------------+---------------+---------------+---------------+----------+-----------+---------------+-------------------+----------------------+----------------------+
|date      |location_key|new_confirmed|cumulative_confirmed|new_confirmed_male|new_confirmed_female|cumulative_confirmed_male|cumulative_confirmed_female|country_name|aggregation_level|iso_3166_1_alpha_2|iso_3166_1_alpha_3|subregion1_code|subregion1_name|subregion2_code|subregion2_name|population|gdp_usd    |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|
+----------+------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+------------+-----------------+------------------+----------

25/06/05 22:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:23:06 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:23:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


##### **2.** Left Join:

In [35]:
df_left_join_countries_cases = df_countries.join(
    df_cases, on=["date", "location_key"], how="left"
)

df_left_join_countries_cases.show(10, truncate=False)

df_left_join_countries_cases.write.mode("overwrite").parquet(
    "output/left_join_countries_cases.parquet"
)

+----------+------------+--------------+-----------------+------------------+------------------+---------------+-------------------+---------------+-------------------+----------+---------+---------------+-------------------+----------------------+----------------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+
|date      |location_key|country_name  |aggregation_level|iso_3166_1_alpha_2|iso_3166_1_alpha_3|subregion1_code|subregion1_name    |subregion2_code|subregion2_name    |population|gdp_usd  |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|new_confirmed|cumulative_confirmed|new_confirmed_male|new_confirmed_female|cumulative_confirmed_male|cumulative_confirmed_female|
+----------+------------+--------------+-----------------+------------------+------------------+---------------+-------------------+---------------+-------------------+----------+---------+---------

25/06/05 22:23:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:23:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:23:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


##### **3.** Right Join:

In [36]:
df_right_join_countries_disorders = df_countries.join(
    df_disorders, on=["date", "location_key"], how="right"
)

df_right_join_countries_disorders.show(10, truncate=False)

df_right_join_countries_disorders.write.mode("overwrite").parquet(
    "output/right_join_countries_disorders.parquet"
)

+----------+------------+------------------------+-----------------+------------------+------------------+---------------+---------------+---------------+-----------------+----------+-------+---------------+-------------------+----------------------+----------------------+------------------------+---------------------+------------------------+----------------------+
|date      |location_key|country_name            |aggregation_level|iso_3166_1_alpha_2|iso_3166_1_alpha_3|subregion1_code|subregion1_name|subregion2_code|subregion2_name  |population|gdp_usd|nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|search_trends_alcoholism|search_trends_anxiety|search_trends_depression|search_trends_insomnia|
+----------+------------+------------------------+-----------------+------------------+------------------+---------------+---------------+---------------+-----------------+----------+-------+---------------+-------------------+----------------------+------------

25/06/05 22:24:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:24:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


##### **4.** Full Join:

In [37]:
df_full_join_cases_deceased = df_cases.join(
    df_deceased, on=["date", "location_key"], how="full"
)

df_full_join_cases_deceased.show(10, truncate=False)

df_full_join_cases_deceased.write.mode("overwrite").parquet(
    "output/full_join_cases_deceased.parquet"
)

+----------+------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+------------+-------------------+-----------------+-------------------+------------------------+--------------------------+
|date      |location_key|new_confirmed|cumulative_confirmed|new_confirmed_male|new_confirmed_female|cumulative_confirmed_male|cumulative_confirmed_female|new_deceased|cumulative_deceased|new_deceased_male|new_deceased_female|cumulative_deceased_male|cumulative_deceased_female|
+----------+------------+-------------+--------------------+------------------+--------------------+-------------------------+---------------------------+------------+-------------------+-----------------+-------------------+------------------------+--------------------------+
|2020-01-01|AF          |0            |0                   |NULL              |NULL                |NULL                     |NULL                       |0           

25/06/05 22:25:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


##### **5.** Cross Join:

In [38]:
df_poland_last_5_records = df_countries.filter(
    (df_countries["country_name"] == "Poland")
).orderBy("date", ascending=False).limit(5)

df_us_last_5_records = df_countries.filter(
    (df_countries["country_name"] == "United States of America")
).orderBy("date", ascending=False).limit(5)

df_poland_prefix = add_col_prefix(df_poland_last_5_records, "poland")
df_us_prefix = add_col_prefix(df_us_last_5_records, "us")

df_cross_join_countries_countries = df_poland_prefix.crossJoin(df_us_prefix)

df_cross_join_countries_countries.show(10, truncate=False)

df_cross_join_countries_countries.write.mode("overwrite").parquet(
    "output/cross_join_countries_countries.parquet"
)

+-------------------+-----------+-------------------+------------------------+-------------------------+-------------------------+----------------------+----------------------+----------------------+----------------------+-----------------+--------------+----------------------+--------------------------+-----------------------------+-----------------------------+---------------+----------+------------------------+--------------------+---------------------+---------------------+------------------+------------------+------------------+------------------+-------------+----------+------------------+----------------------+-------------------------+-------------------------+
|poland_location_key|poland_date|poland_country_name|poland_aggregation_level|poland_iso_3166_1_alpha_2|poland_iso_3166_1_alpha_3|poland_subregion1_code|poland_subregion1_name|poland_subregion2_code|poland_subregion2_name|poland_population|poland_gdp_usd|poland_nurses_per_1000|poland_physicians_per_1000|poland_hospital_b

##### **6.** Semi Join:

In [39]:
df_semi_join_countries_vaccinations = df_countries.join(
    df_vaccinations, on=["date", "location_key"], how="semi"
)

df_semi_join_countries_vaccinations.show(10, truncate=False)

df_semi_join_countries_vaccinations.write.mode("overwrite").parquet(
    "output/semi_join_countries_vaccinations.parquet"
)

+----------+------------+------------+-----------------+------------------+------------------+---------------+----------------------+---------------+---------------+----------+-------------+---------------+-------------------+----------------------+----------------------+
|date      |location_key|country_name|aggregation_level|iso_3166_1_alpha_2|iso_3166_1_alpha_3|subregion1_code|subregion1_name       |subregion2_code|subregion2_name|population|gdp_usd      |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|
+----------+------------+------------+-----------------+------------------+------------------+---------------+----------------------+---------------+---------------+----------+-------------+---------------+-------------------+----------------------+----------------------+
|2020-02-26|BR          |Brazil      |0                |BR                |BRA               |NULL           |NULL                  |NULL           |NULL           |212559409 |18397

25/06/05 22:26:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


##### **7.** Anti Join:

In [40]:
df_anti_join_countries_vaccinations = df_countries.join(
    df_vaccinations, on=["date", "location_key"], how="anti"
)

df_anti_join_countries_vaccinations.show(10, truncate=False)

df_anti_join_countries_vaccinations.write.mode("overwrite").parquet(
    "output/anti_join_countries_vaccinations.parquet"
)

+----------+------------+------------+-----------------+------------------+------------------+---------------+---------------------+---------------+----------------------+----------+-----------+---------------+-------------------+----------------------+----------------------+
|date      |location_key|country_name|aggregation_level|iso_3166_1_alpha_2|iso_3166_1_alpha_3|subregion1_code|subregion1_name      |subregion2_code|subregion2_name       |population|gdp_usd    |nurses_per_1000|physicians_per_1000|hospital_beds_per_1000|health_expenditure_usd|
+----------+------------+------------+-----------------+------------------+------------------+---------------+---------------------+---------------+----------------------+----------+-----------+---------------+-------------------+----------------------+----------------------+
|2020-01-01|AF          |Afghanistan |0                |AF                |AFG               |NULL           |NULL                 |NULL           |NULL                 

25/06/05 22:27:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
25/06/05 22:27:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


## Część 3
**Misja dodatkowa - Tworzenie i wykorzystanie funkcji UDF**

#### **3.1.** Zdefiniuj własną funkcję klasyfikującą poziom zagrożenia epidemiologicznego w oparciu o liczbę potwierdzonych przypadków COVID-19. Określ trzy poziomy zagrożenia: niski, średni i wysoki. Ustal własne progi klasyfikacji i uzasadnij wybór wartości progowych.

Przykładowe progi (na miesiąc):
- low – poniżej 1 000 przypadków,
- medium – od 1 000 do 10 000 przypadków,
- high – powyżej 10 000 przypadków.

In [41]:
epidemic_risk_thresholds = {
    'low': (-float("inf"), 999),
    'medium': (1000, 9999),
    'high': (10000, float("inf"))
}

def classify_epidemic_risk(num_confirmed):
    if not isinstance(num_confirmed, (int, float)):
        return 'unknown'

    for risk_threshold, (lower_bound, upper_bound) in epidemic_risk_thresholds.items():
        if lower_bound <= num_confirmed <= upper_bound:
            return risk_threshold

    return 'unknown'


#### **3.2.** Zamień zdefiniowaną funkcję na UDF czyli funkcję użytkownika kompatybilną z PySparkiem z wykorzystaniem pyspark.sql.functions.udf oraz StringType() z pyspark.sql.types. Funkcja ta powinna działać na kolumnach DataFrame i zwracać nową wartość dla każdego wiersza.

In [42]:
classify_epidemic_risk_udf = F.udf(classify_epidemic_risk, T.StringType())

#### **3.3.** Dodaj nową kolumnę do DataFrame, która będzie zawierać przypisany poziom zagrożenia dla każdego rekordu. Wykorzystaj utworzoną funkcję UDF oraz funkcję do tworzenia kolumn (withColumn) w PySparku.

In [43]:
df_combined = df_combined.withColumn(
    "epidemic_risk_level",
    classify_epidemic_risk_udf(df_combined['new_confirmed'])
)

#### **3.4.** Wyświetl tabelę zawierającą co najmniej trzy kolumny: nazwę kraju, liczbę potwierdzonych przypadków oraz poziom zagrożenia. Sprawdź, czy wartości przypisane przez UDF są zgodne z Twoimi progami.

In [44]:
df_combined.select(
    'country_name', 'new_confirmed', 'epidemic_risk_level'
).show(10, truncate=False)

df_combined.write.mode("overwrite").parquet("output/combined_with_risk_level.parquet")

+------------+-------------+-------------------+
|country_name|new_confirmed|epidemic_risk_level|
+------------+-------------+-------------------+
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
|Andorra     |0            |low                |
+------------+-------------+-------------------+
only showing top 10 rows


#### **3.5.** Przeprowadź grupowanie danych według poziomu zagrożenia i zlicz, ile rekordów należy do każdej kategorii (groupBy() i count()). Odpowiedz, czy występuje przewaga jednego z poziomów zagrożenia i co może to oznaczać dla dalszej analizy.

In [45]:
df_combined.groupBy('epidemic_risk_level').agg(
    F.count(F.lit(1)).alias('records_count')
).show(truncate=False)

+-------------------+-------------+
|epidemic_risk_level|records_count|
+-------------------+-------------+
|low                |185422       |
|unknown            |16009        |
|high               |10989        |
|medium             |31567        |
+-------------------+-------------+



## Część 4

Operacja map bierze funkcję jako argument i stosuje tę funkcję do każdego elementu RDD lub DF, zwracając nowy RDD lub DF, który zawiera przekształcone elementy. Zmapuj swój DF przekształcając wybrane dane (np. zmień format albo wykonaj dowolne obliczenia).

In [46]:
from pyspark.sql.types import MapType


def map_daily_stats_to_json(row):
    return {
        "body": {
            "date": row.date,
            "country_name": row.country_name,
            "new_confirmed": row.new_confirmed,
            "cumulative_confirmed": row.cumulative_confirmed,
            "new_deceased": row.new_deceased,
            "cumulative_deceased": row.cumulative_deceased,
            "new_persons_vaccinated": row.new_persons_vaccinated,
            "cumulative_persons_vaccinated": row.cumulative_persons_vaccinated,
            "search_trends_depression": row.search_trends_depression
        }
    }

schema_json = StructType([
    StructField("body", MapType(StringType(), StringType()), False)
])

json_df_daily_stats = df_combined.rdd.map(map_daily_stats_to_json).toDF(schema_json)


json_df_daily_stats.show(5, truncate=False)

json_df_daily_stats.write.mode("overwrite").json("output/daily_stats.json")

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|body                                                                                                                                                                                                                                              |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{new_persons_vaccinated -> NULL, date -> 2020-01-01, new_deceased -> 0, cumulative_deceased -> 0, new_confirmed -> 0, country_name -> Andorra, search_trends_depression -> NULL, cumulative_persons_vaccinated -> NULL, cumulative_confirmed -> 0}|
|{new_persons_vaccin